# 电商用户画像与价值分层｜来源代码整理

原作者：**代维**；来源：[和鲸社区原项目](https://www.heywhale.com/mw/project/69987eec9bb7c0d2d559ed2e)。

本 Notebook 从用户提供的粘贴文本整理代码，保留原始逻辑，不是原作者的原始 ipynb 文件，也不是本次已执行的分析结果。代码输出保持为空，尚未取得原始 CSV，未执行。

运行前请安装同目录 requirements.txt 所列依赖，并修改下一单元的 `/home/mw/input/...` 路径。原始代码仍包含中文字体配置、年龄分箱和同值分配等待核对问题，详见 `../docs/material_review.md`。R 使用登录间隔，不是最近购买间隔。


## 环境与数据读取

代码来自 `source/provided_project.txt` 第72至85行，按原样提取。


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from scipy.stats import gaussian_kde
# 设置中文显示，避免图表乱码
plt.rcParams['font.sans-serif'] = ['SimHei']  #  Windows用这个
# plt.rcParams['font.sans-serif'] = ['Arial Unicode MS']  # Mac用这个，注释掉上面一行，打开这行
plt.rcParams['axes.unicode_minus'] = False
df = pd.read_csv('/home/mw/input/01236996/user_personalized_features.csv')
print("数据基本信息：")
print(df.info())
print("\n数据描述性统计：")
print(df.describe())


## 缺失值检查

代码来自 `source/provided_project.txt` 第139至140行，按原样提取。


In [ ]:
print("\n各字段缺失值数量：")
print(df.isnull().sum())


## 重复行检查

代码来自 `source/provided_project.txt` 第158至159行，按原样提取。


In [ ]:
print("\n重复数据行数：", df.duplicated().sum())
df = df.drop_duplicates()


## 筛选与清洗

代码来自 `source/provided_project.txt` 第161至165行，按原样提取。


In [ ]:
df = df[(df['Age'] >= 18) & (df['Age'] <= 70)]  # 只保留18-70岁的合理用户
df = df[df['Purchase_Frequency'] >= 0]  # 购买频率不能为负
df = df[df['Total_Spending'] >= 0]  # 消费额不能为负
df = df.reset_index(drop=True)
print(f"\n清洗后数据量：{df.shape[0]}行，{df.shape[1]}列")


## 用户性别、年龄、地域与兴趣画像

代码来自 `source/provided_project.txt` 第167至213行，按原样提取。


In [ ]:
# 1. 性别分布
plt.figure(figsize=(8, 5))
gender_count = df['Gender'].value_counts()
sns.barplot(x=gender_count.index, y=gender_count.values, palette='Blues_d')
plt.title('平台用户性别分布', fontsize=14)
plt.xlabel('性别', fontsize=12)
plt.ylabel('用户数量', fontsize=12)
# 给柱子加数值标签
for i, v in enumerate(gender_count.values):
    plt.text(i, v+5, str(v), ha='center', fontsize=10)
plt.savefig('用户性别分布.png', dpi=300, bbox_inches='tight')
plt.show()

# 2. 年龄分层分布（把年龄分组，更有业务意义）
df['Age_Group'] = pd.cut(df['Age'], bins=[18, 25, 35, 45, 55, 70], labels=['18-25岁', '26-35岁', '36-45岁', '46-55岁', '56岁以上'])
plt.figure(figsize=(10, 5))
age_count = df['Age_Group'].value_counts().sort_index()
sns.barplot(x=age_count.index, y=age_count.values, palette='Purples_d')
plt.title('平台用户年龄分层分布', fontsize=14)
plt.xlabel('年龄分组', fontsize=12)
plt.ylabel('用户数量', fontsize=12)
for i, v in enumerate(age_count.values):
    plt.text(i, v+5, str(v), ha='center', fontsize=10)
plt.savefig('用户年龄分布.png', dpi=300, bbox_inches='tight')
plt.show()

# 3. 用户地域分布
plt.figure(figsize=(8, 5))
location_count = df['Location'].value_counts()
sns.barplot(x=location_count.index, y=location_count.values, palette='Greens_d')
plt.title('平台用户地域分布', fontsize=14)
plt.xlabel('居住地', fontsize=12)
plt.ylabel('用户数量', fontsize=12)
for i, v in enumerate(location_count.values):
    plt.text(i, v+5, str(v), ha='center', fontsize=10)
plt.savefig('用户地域分布.png', dpi=300, bbox_inches='tight')
plt.show()

# 4. 用户兴趣分布
plt.figure(figsize=(12, 6))
interest_count = df['Interests'].value_counts()
sns.barplot(x=interest_count.values, y=interest_count.index, palette='Oranges_d')
plt.title('平台用户兴趣分布', fontsize=14)
plt.xlabel('用户数量', fontsize=12)
plt.ylabel('兴趣标签', fontsize=12)
plt.savefig('用户兴趣分布.png', dpi=300, bbox_inches='tight')
plt.show()


## 登录分布与邮件订阅对比

代码来自 `source/provided_project.txt` 第343至366行，按原样提取。


In [ ]:
# 1. 上次登录天数分布（判断用户活跃/流失情况）
# 提取数据并去除空值
data = df['Last_Login_Days_Ago'].dropna()
plt.figure(figsize=(10, 5))
# 绘制直方图，和原需求效果完全一致
n, bins, patches = plt.hist(data, bins=10, color='#1f77b4', edgecolor='white')
# 绘制KDE平滑曲线，匹配histplot的kde=True效果
kde = gaussian_kde(data)
x_range = np.linspace(bins.min(), bins.max(), 1000)
# 缩放KDE曲线，匹配直方图的用户数量刻度
plt.plot(x_range, kde(x_range) * len(data) * np.diff(bins)[0], color='#1f77b4', linewidth=2)

plt.title('用户上次登录距今天数分布', fontsize=14)
plt.xlabel('距上次登录天数', fontsize=12)
plt.ylabel('用户数量', fontsize=12)
plt.show()
# 2. 订阅用户vs非订阅用户的活跃度对比
plt.figure(figsize=(10, 5))
sns.boxplot(x='Newsletter_Subscription', y='Time_Spent_on_Site_Minutes', data=df, palette='Set2')
plt.title('邮件订阅用户与非订阅用户的网站停留时长对比', fontsize=14)
plt.xlabel('是否订阅邮件', fontsize=12)
plt.ylabel('网站停留时长（分钟）', fontsize=12)
plt.savefig('订阅用户活跃度对比.png', dpi=300, bbox_inches='tight')
plt.show()


## 品类偏好与年龄层消费

代码来自 `source/provided_project.txt` 第500至520行，按原样提取。


In [ ]:
# 1. 产品品类偏好分布
plt.figure(figsize=(12, 6))
category_count = df['Product_Category_Preference'].value_counts()
sns.barplot(x=category_count.values, y=category_count.index, palette='RdBu_d')
plt.title('用户产品品类偏好分布', fontsize=14)
plt.xlabel('用户数量', fontsize=12)
plt.ylabel('产品品类', fontsize=12)
plt.savefig('品类偏好分布.png', dpi=300, bbox_inches='tight')
plt.show()

# 2. 不同年龄层的总消费均值对比
plt.figure(figsize=(10, 5))
age_spending = df.groupby('Age_Group')['Total_Spending'].mean().sort_index()
sns.barplot(x=age_spending.index, y=age_spending.values, palette='coolwarm')
plt.title('不同年龄层用户平均总消费', fontsize=14)
plt.xlabel('年龄分组', fontsize=12)
plt.ylabel('平均总消费（元）', fontsize=12)
for i, v in enumerate(age_spending.values):
    plt.text(i, v+50, f'{int(v)}元', ha='center', fontsize=10)
plt.savefig('年龄层消费对比.png', dpi=300, bbox_inches='tight')
plt.show()


## 以登录新近度替代R的八类价值分层

代码来自 `source/provided_project.txt` 第615至669行，按原样提取。


In [ ]:
# 1. 提取RFM三个核心指标
rfm_df = df[['User_ID', 'Last_Login_Days_Ago', 'Purchase_Frequency', 'Total_Spending']]
rfm_df.columns = ['User_ID', 'R', 'F', 'M']

# 2. 给RFM指标打分（1-5分，5分为最优）
# R值：越小越好，所以倒序打分
rfm_df['R_Score'] = pd.qcut(rfm_df['R'].rank(method='first'), q=5, labels=[5,4,3,2,1])
# F值：越大越好，正序打分
rfm_df['F_Score'] = pd.qcut(rfm_df['F'].rank(method='first'), q=5, labels=[1,2,3,4,5])
# M值：越大越好，正序打分
rfm_df['M_Score'] = pd.qcut(rfm_df['M'].rank(method='first'), q=5, labels=[1,2,3,4,5])

# 3. 拼接RFM得分，生成用户标签
rfm_df['RFM_Score'] = rfm_df['R_Score'].astype(str) + rfm_df['F_Score'].astype(str) + rfm_df['M_Score'].astype(str)

# 4. 用户价值分层（经典8分法，简化版，容易理解和讲解）
def user_level(row):
    r, f, m = row['R_Score'], row['F_Score'], row['M_Score']
    if r >=4 and f >=4 and m >=4:
        return '高价值用户'
    elif r >=4 and f >=4 and m <4:
        return '潜力用户'
    elif r >=4 and f <4 and m >=4:
        return '深耕用户'
    elif r >=4 and f <4 and m <4:
        return '新用户'
    elif r <4 and f >=4 and m >=4:
        return '流失预警用户'
    elif r <4 and f >=4 and m <4:
        return '需召回用户'
    elif r <4 and f <4 and m >=4:
        return '流失高价值用户'
    else:
        return '低价值用户'

rfm_df['User_Level'] = rfm_df.apply(user_level, axis=1)

# 5. 查看各用户群体数量分布
level_count = rfm_df['User_Level'].value_counts()
print("\n用户价值分层结果：")
print(level_count)

# 6. 可视化用户分层结果
plt.figure(figsize=(12, 6))
sns.barplot(x=level_count.values, y=level_count.index, palette='Spectral')
plt.title('RFM用户价值分层分布', fontsize=14)
plt.xlabel('用户数量', fontsize=12)
plt.ylabel('用户分层', fontsize=12)
for i, v in enumerate(level_count.values):
    plt.text(v+5, i, str(v), va='center', fontsize=10)
plt.savefig('RFM用户分层.png', dpi=300, bbox_inches='tight')
plt.show()

# 7. 把分层结果合并回原数据，方便后续分析
df = pd.merge(df, rfm_df[['User_ID', 'User_Level']], on='User_ID', how='left')


## 指标相关性热力图

代码来自 `source/provided_project.txt` 第859至873行，按原样提取。


In [ ]:
# 筛选数值型字段，计算和总消费的相关性
num_cols = ['Age', 'Income', 'Last_Login_Days_Ago', 'Purchase_Frequency', 
            'Average_Order_Value', 'Time_Spent_on_Site_Minutes', 'Pages_Viewed', 'Total_Spending']
corr_df = df[num_cols].corr()

# 可视化相关性热力图
plt.figure(figsize=(10, 8))
sns.heatmap(corr_df, annot=True, cmap='coolwarm', vmin=-1, vmax=1, fmt='.2f')
plt.title('用户指标相关性热力图', fontsize=14)
plt.savefig('相关性热力图.png', dpi=300, bbox_inches='tight')
plt.show()

# 输出和总消费相关性最高的指标
print("\n和用户总消费相关性TOP5指标：")
print(corr_df['Total_Spending'].sort_values(ascending=False).head(6))


## 结果状态

原文附带的八类人数合计为1000，但并非本次重跑验证结果。原文部分结论与输出不一致；请先核对原数据，再补充运行结果。没有原CSV时，不执行这些代码。
